In [ ]:
# @title Options Scanner: Top 20 "Value Zone" Candidates
# @markdown Logic: Dip (3 Candles) + Price < Mid BB + Stoch Turn + **Trend Rank**

import sys
import os
import warnings
import logging
import yfinance as yf
import pandas as pd
import pandas_ta as ta

# 1. SETUP & SUPPRESSION
print("Installing libraries and configuring environment...")
!pip install yfinance pandas_ta -q

class SuppressStderr:
    def __enter__(self):
        self.errnull_file = open(os.devnull, 'w')
        self.old_stderr = sys.stderr
        sys.stderr = self.errnull_file
    def __exit__(self, *_):
        sys.stderr = self.old_stderr
        self.errnull_file.close()

warnings.filterwarnings("ignore")
logging.getLogger('yfinance').setLevel(logging.CRITICAL)

# 2. WATCHLIST
ticker_string = """
DI ADM ADP ADSK AEP AES AFL AFRM AGNC AI AIG AKAM ALB ALK ALL AMAT AMD AMGN AMRN AMZN APA APH APTV AVGO AXP BA BABA BAC BAX BBY BEN BIDU BIIB BITO BK BKR BMY BP BSX BX BYND C CAH CAT CB CCI CCJ CCL CF CFG CHWY CI CL CLF CLX CMCSA CME CNC CNP COF COIN COP COST CPB CPRI CRM CRON CRWD CSCO CSX CTVA CVNA CVS CVX CZR D DAL DD DE DELL DHI DHR DIA DIS DKNG DLR DOCU DOW DRI DVN DXC EA EBAY ED EEM EFA EIX EL EMR ENPH EOG EQR EQT ETSY EVRG EW EWJ EWW EWY EWZ EXC EXPE F FANG FAST FCX FDX FE FI FITB FOXA FSLR FTI FTV FXE FXI GD GDX GE GILD GLD GLW GM GME GOOG GOOGL GPRO GPS GS HAL HBAN HBI HCA HD HIG HLT HOG HOLX HON HPE HPQ HYG IBB IBIT IBM ICE IEF INTC IP IPG IRM IVZ IWM IYR JCI JD JETS JNJ JNPR JPM K KHC KMI KO KR KRE KSS KWEB LEN LI LKQ LLY LNC LOW LQD LUMN LUV LVS LYB LYFT MA MAR MARA MAS MCD MCHP MDLZ MDT MET META MGM MMM MO MOS MPC MRK MRNA MRVL MS MSFT MSTR MTB MTCH MU NCLH NEE NEM NET NFLX NKE NOW NRG NTAP NTES NVAX NVDA NWL NWSA O OIH OKE OMC ON ORCL OXY PARA PBR PDD PENN PEP PFE PFG PG PGR PINS PLTR PNC PPL PRU PSX PTON PYPL QCOM QQQ RBLX RCL RF RIG RIOT RIVN ROKU ROST RTX RUN SBUX SCHD SCHW SEDG SHOP SIRI SLB SLV SMCI SMH SNAP SNOW SO SOFI SOXL SOXS SPG SPX SPXL SPXS SPY SQQQ STX SWKS SYF SYY T TAP TBT TCOM TDOC TFC TGT TJX TLT TMO TMUS TPR TQQQ TRIP TRV TSLA TSM TSN TT TTD TTWO TXN U UA UAA UAL UBER ULTA UNG UNH UNP UPS UPST URBN USB USO UVXY V VALE VFC VGK VTR VXX VZ WAB WBA WBD WDC WFC WM WMB WMT WU WY WYNN X XBI XEL XHB XLB XLC XLE XLF XLI XLK XLP XLRE XLU XLV XLY XOM XOP XRT XRX XSP XYZ YELP YINN ZION ZM
"""
tickers = [t.strip() for t in ticker_string.replace('\n', ' ').split(' ') if t.strip() != '']

candidates = []

print(f"Scanning {len(tickers)} stocks on HOURLY timeframe...")
print("1. Dip Logic: Price touched Lower BB in last 3 hours.")
print("2. Value Logic: Price is < Middle BB (Still discounted).")
print("3. Ranking: Sorted by 200 SMA Trend Strength.")
print("-" * 60)

# 3. SCAN LOOP
with SuppressStderr():
    for symbol in tickers:
        try:
            # We need ~3mo of data to calculate a valid 200 SMA on hourly chart
            df = yf.download(symbol, period="3mo", interval="1h", progress=False, auto_adjust=True)

            if df.empty or len(df) < 200: continue

            # Flatten MultiIndex
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            # --- INDICATORS ---
            # BB (20, 2)
            bb = df.ta.bbands(length=20, std=2)
            if bb is None: continue
            df = pd.concat([df, bb], axis=1)

            # Stoch (14, 3, 3)
            stoch = df.ta.stoch(k=14, d=3, smooth_k=3)
            if stoch is None: continue
            df = pd.concat([df, stoch], axis=1)

            # SMA 200
            sma200 = df.ta.sma(length=200)
            if sma200 is None: continue
            df = pd.concat([df, sma200], axis=1)

            # Define Columns dynamically
            bbl_col = [c for c in df.columns if 'BBL' in c][0]
            bbm_col = [c for c in df.columns if 'BBM' in c][0] # Middle Band
            stoch_k_col = [c for c in df.columns if 'STOCHk' in c][0]
            stoch_d_col = [c for c in df.columns if 'STOCHd' in c][0]
            sma_col = [c for c in df.columns if 'SMA_200' in c][0]

            # --- WINDOW (Last 4 candles to analyze the last 3 closed) ---
            window = df.iloc[-4:]
            curr_price = window['Close'].iloc[-1]
            curr_k = window[stoch_k_col].iloc[-1]
            curr_d = window[stoch_d_col].iloc[-1]
            curr_bbl = window[bbl_col].iloc[-1]
            curr_bbm = window[bbm_col].iloc[-1]
            curr_sma = window[sma_col].iloc[-1]

            # --- FILTER 1: STOCHASTIC ---
            # K crossed 20 OR (K < 30 AND K > D)
            stoch_passed = False
            k_values = window[stoch_k_col].values
            for i in range(1, len(k_values)):
                if k_values[i-1] < 20 and k_values[i] > 20:
                    stoch_passed = True
            if curr_k < 30 and curr_k > curr_d:
                 stoch_passed = True

            # --- FILTER 2: BOLLINGER DIP & CEILING ---
            bb_passed = False

            # Check 1: Did we touch the bottom in the last 3 candles?
            recent_lows = window['Low'].iloc[-3:].values
            recent_bbls = window[bbl_col].iloc[-3:].values
            touched_bottom = False
            for i in range(len(recent_lows)):
                if recent_lows[i] < recent_bbls[i]:
                    touched_bottom = True

            # Check 2: Price Ceiling (Must be BELOW Middle Band)
            below_mid = curr_price < curr_bbm

            # Check 3: Price Floor (Must be ABOVE Lower Band - i.e., not currently crashing)
            above_low = curr_price > curr_bbl

            if touched_bottom and below_mid and above_low:
                bb_passed = True

            # --- COLLECT CANDIDATES ---
            if stoch_passed and bb_passed:
                # Calculate Trend Score: % Distance above/below 200 SMA
                trend_score = ((curr_price - curr_sma) / curr_sma) * 100

                candidates.append({
                    'Ticker': symbol,
                    'Price': curr_price,
                    'Stoch K': curr_k,
                    'Trend Strength': trend_score
                })

        except Exception:
            continue

# 4. RANKING & OUTPUT
if len(candidates) > 0:
    df_res = pd.DataFrame(candidates)

    # SORT: By Trend Strength (Safest uptrends first)
    df_sorted = df_res.sort_values(by='Trend Strength', ascending=False)

    # LIMIT: Top 20
    top_20 = df_sorted.head(20)

    print(f"\nScan Complete. Found {len(df_res)} matches.")
    print("Filtered to TOP 20 ranked by 200 SMA Trend Strength.")
    print("-" * 80)
    print(f"{'Ticker':<10} {'Price':<10} {'Stoch K':<10} {'Trend %':<15} {'Strategy Note'}")
    print("-" * 80)

    for index, row in top_20.iterrows():
        note = "Top Tier" if row['Trend Strength'] > 0 else "Weak Trend"
        print(f"{row['Ticker']:<10} {row['Price']:<10.2f} {row['Stoch K']:<10.2f} {row['Trend Strength']:<15.2f} {note}")

    # Create Comma Separated Lines (10 per line)
    tickers = top_20['Ticker'].tolist()

    line1 = tickers[:10]
    line2 = tickers[10:20]

    print("-" * 80)
    print("\nCOPY FOR TRADINGVIEW (10 per line):")
    if len(line1) > 0: print(",".join(line1))
    if len(line2) > 0: print(",".join(line2))

else:
    print("No stocks matched the criteria.")